# EDA Precinct6 chica — `witfoo/precinct6-cybersecurity` (2M)

Objetivo: describir **todas las variables interesantes** para decidir modelo después.
Guía proyecto: fenómeno SOC, 3 prepros, 3 modelos + complejidad, deploy.

**Fuentes:**
- Dataset 2M: https://huggingface.co/datasets/witfoo/precinct6-cybersecurity
- Dataset 114M: https://huggingface.co/datasets/witfoo/precinct6-cybersecurity-100m
- Código generación: https://github.com/witfoo/dataset-from-precinct6
- Research: https://witfoo.com/research
- ATT&CK: https://attack.mitre.org/ | NodLink: https://arxiv.org/abs/2311.02331 | CybORG: https://arxiv.org/abs/2002.10667
- EDA SOC: imbalance manda (DeepCASE https://arxiv.org/html/2507.01571), usar PR-AUC no accuracy, stratified k-fold, cost-sensitive/focal loss, tuning reglas.

In [ ]:
import json, pandas as pd, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path

plt.rcParams['figure.figsize'] = (10, 4)
SIG = Path('data/signals/signals.parquet')
INC = Path('data/signals/incident_signals.parquet')
NODES = Path('data/graph/nodes.parquet')
EDGES = Path('data/graph/edges-00000.parquet')
REPORTS = Path('data/graph/attack_reports.parquet')
print('archivos:', [p.exists() for p in [SIG, INC, NODES, EDGES, REPORTS]])

## 1. Overview signals (2,011,674 x 38)
38 columnas: timestamp, event_time, timestamp_source, origin, org_id, artifact_id, message_type, stream_name, pipeline, src/dst_ip, src/dst_port, protocol, src/dst_host, username, action, severity, vendor_code, message_sanitized, label_binary, label_confidence, attack_techniques/tactics, defense_techniques, mo_name, suspicion_score, lifecycle_stage, disposition, disposition_category, is_false_positive, status_name, incident_ids, matched_rules, set_roles, product_name, vendor_name.

In [ ]:
df = pd.read_parquet(SIG)
print(df.shape)
print(df.dtypes)
print('\nmiss %:')
print((df.isna().sum() / len(df) * 100).sort_values(ascending=False).head(12))
print('\nvacíos string %:')
print(((df == '').sum() / len(df) * 100).sort_values(ascending=False).head(12))
print('\nduplicados artifact_id:', df.artifact_id.duplicated().sum())
print('memoria MB:', df.memory_usage(deep=True).sum() / 1e6)

## 2. Target: label_binary (desbalance extremo)
Esperado README: benign 94.31% / suspicious 5.31% / malicious 0.38%. IR ~245:1. Esto define todo el modelado.

In [ ]:
print(df.label_binary.value_counts())
print(df.label_binary.value_counts(normalize=True) * 100)
df.label_binary.value_counts().plot(kind='bar'); plt.title('label_binary'); plt.show()
print('\nlabel_confidence por label:')
print(df.groupby('label_binary').label_confidence.describe())
df.hist(column='label_confidence', by='label_binary', bins=20); plt.show()
print('\nsuspicion_score por label:')
print(df.groupby('label_binary').suspicion_score.describe())

## 3. Temporal: timestamp vs event_time
Live 2024-07-26 → 2024-08-01. timestamp=ingesta Precinct, event_time=device clock (muchos NaN). Revisar lag y cobertura por día/hora y por label.

In [ ]:
df['ts'] = pd.to_datetime(df.timestamp, unit='s', utc=True)
print(df.ts.min(), df.ts.max())
print('event_time NaN %:', df.event_time.isna().mean() * 100)
df['lag_h'] = (df.timestamp - df.event_time) / 3600
print(df.lag_h.describe())
df.set_index('ts').resample('6h').size().plot(); plt.title('eventos por 6h'); plt.show()
df['hour'] = df.ts.dt.hour
pd.crosstab(df.hour, df.label_binary).plot(kind='bar', stacked=True); plt.title('hora x label'); plt.show()
print(df.groupby([df.ts.dt.date, 'label_binary']).size().unstack(fill_value=0))

## 4. Orgs (cobertura desigual)
README: ORG-0004 996k, ORG-0005 577k, ORG-0003 434k, ORG-0001 5k. Malicious solo en 0004/0005/0003.

In [ ]:
print(pd.crosstab(df.org_id, df.label_binary))
print(pd.crosstab(df.org_id, df.label_binary, normalize='index') * 100)
print('\ntimestamp_source:', df.timestamp_source.value_counts())

## 5. Productos / streams / pipelines
Top streams: aws_cloudtrail_events, microsoft-windows-security-auditing, windows_security_audit, vcenter, cisco_asa... Ver cuáles concentran malicious.

In [ ]:
for c in ['stream_name', 'product_name', 'vendor_name', 'pipeline', 'message_type']:
    print(f'\n=== {c} top10 ==='); print(df[c].value_counts().head(10))
print('\n--- message_type x label (top malicious) ---')
print(pd.crosstab(df.message_type, df.label_binary)['malicious'].sort_values(ascending=False).head(15))
print('\n--- stream x label malicious share ---')
ct = pd.crosstab(df.stream_name, df.label_binary, normalize='index') * 100
print(ct.sort_values('malicious', ascending=False).head(10))

## 6. Red: protocolo, puertos, hosts, IPs
protocol 6=TCP 17=UDP 1=ICMP. IPs sanitizadas (HMAC). Cardinalidad hosts = feature grafo.

In [ ]:
print('protocol:', df.protocol.value_counts().head())
print('\ndst_port top:', df.dst_port.value_counts().head(10))
print('\nsrc_port top:', df.src_port.value_counts().head(10))
for c in ['src_host', 'dst_host', 'src_ip', 'dst_ip', 'username']:
    print(c, 'nunique:', df[c].nunique(), '| vacíos %:', (df[c] == '').mean() * 100)
print('\nTop src_host:', df.src_host.value_counts().head(5).to_dict())
print('Top dst_port en malicious:', df[df.label_binary == 'malicious'].dst_port.value_counts().head(10).to_dict())

## 7. Acción / severidad / usuario
action: block/permit/logon... severity: warning/info... username USER-NNNN compartido con nodos cred.

In [ ]:
print('action:', df.action.value_counts().head(10).to_dict())
print('severity:', df.severity.value_counts().head(10).to_dict())
print(pd.crosstab(df.action, df.label_binary))
print('\nvendor_code top:', df.vendor_code.value_counts().head(10).to_dict())
print('\nusername vacío % por label:')
print(df.assign(nouser=df.username == '').groupby('label_binary').nouser.mean() * 100)

## 8. ATT&CK + incidente (JSON arrays como string)
attack_techniques ej '["T1041","T1190"]', mo_name ej Data Theft, lifecycle_stage, set_roles, matched_rules. Vacío en benign.

In [ ]:
def top_json(col, n=15):
    c = Counter()
    for v in df.loc[df[col].notna() & (df[col] != ''), col].head(300000):
        try: c.update(json.loads(v))
        except: pass
    return c.most_common(n)

print('mo_name:', df.mo_name.value_counts().head(10).to_dict())
print('lifecycle:', df.lifecycle_stage.value_counts().head(10).to_dict())
print('ATT&CK techniques:', top_json('attack_techniques'))
print('ATT&CK tactics:', top_json('attack_tactics'))
print('defense:', top_json('defense_techniques'))
print('set_roles:', top_json('set_roles'))
print('matched_rules:', top_json('matched_rules'))
print('\nmo_name x label:'); print(pd.crosstab(df.mo_name.fillna('(vacio)'), df.label_binary).sort_values('malicious', ascending=False).head(10))

## 9. Disposition (ground truth = motor Precinct, no analista)
automated: Unprocessed/Disrupted/Open/Cold Case. Analista: Resolved/Dismissed/False Positive (raros).

In [ ]:
print(pd.crosstab(df.disposition, df.label_binary))
print('\ndisposition_category:', pd.crosstab(df.disposition_category, df.label_binary))
print('\nis_false_positive:', pd.crosstab(df.is_false_positive, df.label_binary))

## 10. Texto libre message_sanitized
Log crudo sanitizado. Longitud y tokens separan bien firewall vs windows. Base para modelo 2 (TF-IDF).

In [ ]:
df['msg_len'] = df.message_sanitized.str.len()
print(df.groupby('label_binary').msg_len.describe())
df.boxplot(column='msg_len', by='label_binary'); plt.show()
from collections import Counter as C
import re
toks = C()
for t in df[df.label_binary == 'malicious'].message_sanitized.head(2000): toks.update(re.findall(r'[A-Za-z]{4,}', t.lower()))
print('tokens malicious:', toks.most_common(15))
toks2 = C()
for t in df[df.label_binary == 'benign'].message_sanitized.head(2000): toks2.update(re.findall(r'[A-Za-z]{4,}', t.lower()))
print('tokens benign:', toks2.most_common(15))
print('\n--- ejemplo malicious ---'); print(df[df.label_binary=='malicious'].message_sanitized.iloc[0][:600])
print('\n--- ejemplo benign ---'); print(df[df.label_binary=='benign'].message_sanitized.iloc[0][:600])

## 11. Las otras 4 tablas
incident_signals 238k (100% malicious histórico 2022-2024), nodes 47k, edges 1.7M, reports 51k.

In [ ]:
inc = pd.read_parquet(INC); print('incident_signals:', inc.shape, inc.label_binary.unique())
print('rango:', pd.to_datetime(inc.timestamp, unit='s', utc=True).min(), '→', pd.to_datetime(inc.timestamp, unit='s', utc=True).max())
nodes = pd.read_parquet(NODES); print('\nnodes:', nodes.shape); print(nodes.type.value_counts())
edges = pd.read_parquet(EDGES); print('\nedges:', edges.shape); print(edges.type.value_counts())
rep = pd.read_parquet(REPORTS); print('\nreports:', rep.shape); print(rep.mo_name.value_counts().head(8).to_dict())
rep['len'] = rep.report_text.str.len(); print(rep['len'].describe())
print(rep.report_text.iloc[0][:800])

## 12. Conclusiones para modelado

**Tarea recomendada:** clasificación 3 clases (benign/suspicious/malicious) o binaria malicious vs resto sobre `signals`.
- IR 245:1 → métrica PR-AUC/F1-macro, nunca accuracy. Stratified k-fold temporal (train ≤2024-07-29, test después) por drift.
- 3 prepros: (1) tabular (one-hot message_type/stream/protocol + scaling ports/lag), (2) texto TF-IDF 5k sobre message_sanitized, (3) grafo (grado host/user desde nodes/edges).
- 3 modelos: (a) XGBoost tabular, (b) LogReg TF-IDF, (c) MLP fusión o GNN simple. Complejidad: XGBoost O(T·d·n log n), LogReg O(n·vocab), GNN O(E).
- Deploy: API SOC que recibe log → devuelve label + técnicas ATT&CK + reporte.
- Ojo: labels = motor Precinct (imperfecto), disposition automated; filtrar `is_false_positive`; malicious concentrado ORG-0004/0005.